## Initialization

In [0]:
import pyspark.sql.functions as F
from pyspark.sql.types import StringType
from pyspark.sql.functions import trim, col

## Read bronze table

In [0]:
df = spark.table("workspace.bronze.erp_loc_a101")

## Silver Transformation

### Clean up customer ID

In [0]:
df = df.withColumn("CID", F.regexp_replace(col("CID"), "-",""))
  

### Country normalization

In [0]:
df = (
    df.withColumn(
        "CNTRY",
        F.when(trim(F.upper(col("CNTRY"))).isin("USA", "US"), "United States")
        .when(trim(F.upper(col("CNTRY"))) == "DE", "Germany")
        .when((col("CNTRY") == "") | col("CNTRY").isNull(), "n/a")
        .otherwise(col("CNTRY"))
    )
)


### Rename column

In [0]:
RENAME_MAP = {
    "CID": "customer_id",
    "CNTRY": "country"
}

for old, new in RENAME_MAP.items():
    df = df.withColumnRenamed(old, new)

### Sanity check of dataframe

In [0]:
df.limit(20).display()

customer_id,country
AW00011000,Australia
AW00011001,Australia
AW00011002,Australia
AW00011003,Australia
AW00011004,Australia
AW00011005,Australia
AW00011006,Australia
AW00011007,Australia
AW00011008,Australia
AW00011009,Australia


### Write silver table

In [0]:
df.write.mode("overwrite").format("delta").saveAsTable("workspace.silver.customer_location")

### Sanity check of silver table

In [0]:
%sql
SELECT *
FROM workspace.silver.customer_location
LIMIT 20

customer_id,country
AW00011000,Australia
AW00011001,Australia
AW00011002,Australia
AW00011003,Australia
AW00011004,Australia
AW00011005,Australia
AW00011006,Australia
AW00011007,Australia
AW00011008,Australia
AW00011009,Australia
